# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** A line chart is best because it clearly shows relative growth trajectories for all continents over time. Indexing each continent to 1970 = 100 removes differences in starting population and makes the fastest growth directly comparable.

In [2]:
# Q1
continent_year = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
continent_1970 = continent_year[continent_year['Year'] == 1970].set_index('Continent')['Population']
continent_year['Population index'] = continent_year.apply(
    lambda r: r['Population'] / continent_1970[r['Continent']] * 100, axis=1
)
fastest = (continent_year[continent_year['Year'] == 2022]
           .sort_values('Population index', ascending=False).iloc[0])
palette = {'Africa':'#0072B2','Asia':'#E69F00','Europe':'#B0B0B0',
           'North America':'#56B4E9','Oceania':'#999999','South America':'#CC79A7'}

fig = px.line(continent_year, x='Year', y='Population index', color='Continent',
              color_discrete_map=palette, template='plotly_white',
              title=f"Africa grew fastest: index reached {fastest['Population index']:.0f} by 2022 (1970 = 100)",
              labels={'Population index':'Population (1970 = 100)','Year':'Year'})
fig.update_traces(line_width=3)
fig.update_layout(showlegend=True, legend_title_text='', hovermode='x unified',
                  xaxis=dict(showgrid=False, dtick=10),
                  yaxis=dict(showgrid=False, rangemode='tozero'))
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** A sorted horizontal bar chart is best for ranking the 15 largest population gains and comparing their magnitudes. Colour highlights whether each country is in Africa, while text labels report exact gains and avoid unnecessary axis decoration.

In [3]:
# Q2
df['Population gain'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Population gain').sort_values('Population gain')
top15['Africa?'] = top15['Continent'].eq('Africa').map(
    {True:'Africa', False:'Other continents'}
)
africa_count = (top15['Continent'] == 'Africa').sum()

fig = px.bar(top15, x='Population gain', y='Country', color='Africa?',
             orientation='h', text='Population gain',
             color_discrete_map={'Africa':'#0072B2','Other continents':'#B0B0B0'},
             template='plotly_white',
             title=f"India and China lead: {africa_count} of the top 15 countries are African",
             labels={'Population gain':'Population increase, 1970–2022','Country':''})
fig.update_traces(texttemplate='%{x:.3s}', textposition='outside', cliponaxis=False)
fig.update_layout(showlegend=True, legend_title_text='', xaxis=dict(showgrid=False),
                  yaxis=dict(showgrid=False), margin=dict(l=120, r=60, t=80))
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** A bubble scatterplot shows the relationship between land area and population on two continuous axes, while bubble size adds 2022 population as a second encoded magnitude. A log scale makes the skewed data readable and continent colours reveal whether the relationship differs by region.

In [4]:
# Q3
outlier_names = ['China','India','United States','Russia','Australia','Canada',
                 'Indonesia','Brazil','Nigeria']
palette = {'Africa':'#0072B2','Asia':'#E69F00','Europe':'#B0B0B0',
           'North America':'#56B4E9','Oceania':'#999999','South America':'#CC79A7'}

fig = px.scatter(df, x='Area', y='2022 Population', color='Continent',
                 size='2022 Population', hover_name='Country',
                 log_x=True, log_y=True, color_discrete_map=palette,
                 template='plotly_white',
                 title='Country size alone does not explain population; few outliers break the pattern',
                 labels={'Area':'Land area (km², log scale)',
                         '2022 Population':'2022 population (log scale)'})
fig.update_traces(marker=dict(sizemode='diameter', sizemin=5,
                              line=dict(width=0.5, color='white')))
for _, row in df[df['Country'].isin(outlier_names)].iterrows():
    fig.add_annotation(x=row['Area'], y=row['2022 Population'], text=row['Country'],
                       showarrow=True, arrowhead=2, arrowsize=0.7, ax=12, ay=-18,
                       font=dict(size=10), bgcolor='rgba(255,255,255,0.7)')
fig.update_layout(showlegend=True, legend_title_text='', xaxis=dict(showgrid=False),
                  yaxis=dict(showgrid=False))
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** A violin plot is best because it shows the full density distribution, spread and median pattern across continents rather than only an average. A log scale makes strongly skewed country densities easier to compare.

In [5]:
# Q4
cont_order = df.groupby('Continent')['Density'].median().sort_values(ascending=False).index.tolist()
palette = {'Africa':'#0072B2','Asia':'#E69F00','Europe':'#B0B0B0',
           'North America':'#56B4E9','Oceania':'#999999','South America':'#CC79A7'}

fig = px.violin(df, x='Continent', y='Density', color='Continent',
                box=True, points='outliers', category_orders={'Continent':cont_order},
                color_discrete_map=palette, template='plotly_white',
                title='Asia has the highest typical density, but every continent has crowding extremes',
                labels={'Density':'Population density (people/km²)','Continent':''})
fig.update_traces(meanline_visible=True, marker=dict(size=4, opacity=0.7))
fig.update_layout(showlegend=False, yaxis=dict(type='log', showgrid=False),
                  xaxis=dict(showgrid=False))
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** A choropleth is best because the question is explicitly geographic and percentage change is a country-level value. A diverging, CVD-safe blue-to-orange scale distinguishes shrinkage from growth without using red or green.

In [6]:
# Q5
df['Population change (%)'] = (
    (df['2022 Population'] - df['2000 Population']) /
    df['2000 Population'] * 100
)
limit = float(max(abs(df['Population change (%)'].min()),
                  abs(df['Population change (%)'].max())))

fig = px.choropleth(df, locations='CCA3', locationmode='ISO-3',
                    color='Population change (%)', hover_name='Country',
                    hover_data={'Population change (%)':':.1f','CCA3':False},
                    color_continuous_scale=[
                        [0.0,'#0072B2'],[0.5,'#E6E6E6'],[1.0,'#D55E00']
                    ],
                    range_color=[-limit,limit], template='plotly_white',
                    title='Population grew almost everywhere since 2000, while a few countries shrank sharply',
                    labels={'Population change (%)':'Change, 2000–2022 (%)'})
fig.update_layout(coloraxis_colorbar=dict(title='Change (%)', ticksuffix='%'),
                  margin=dict(l=20,r=20,t=70,b=20))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** A treemap is the clearest two-level view of 2022 population: continent area shows the major groups and country area shows the parts within each continent. Labels and population values make the hierarchy understandable at a glance.

In [7]:
# Q6
palette = {'Africa':'#0072B2','Asia':'#E69F00','Europe':'#B0B0B0',
           'North America':'#56B4E9','Oceania':'#999999','South America':'#CC79A7'}

fig = px.treemap(df, path=['Continent','Country'], values='2022 Population',
                 color='Continent', color_discrete_map=palette,
                 template='plotly_white',
                 title='Asia holds most of the world: roughly 60% of 2022 population')
fig.update_traces(texttemplate='%{label}<br>%{value:.3s}', textfont_size=12,
                  hovertemplate='%{label}<br>%{value:,.0f} people<extra></extra>')
fig.update_layout(margin=dict(l=20,r=20,t=70,b=20))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** A 100% stacked bar chart compares each year’s continental shares while keeping all bars at 100%. It makes the rise of Africa, the decline of Europe and the dominance of Asia easy to see across the 1970–2022 period.

In [8]:
# Q7
years = [1970,1980,1990,2000,2010,2015,2020,2022]
continent_totals = (df_long[df_long['Year'].isin(years)]
                    .groupby(['Year','Continent'], as_index=False)['Population'].sum())
continent_totals['Group'] = continent_totals['Continent'].where(
    continent_totals['Continent'].isin(['Africa','Asia','Europe']),
    'Rest of the world'
)
shares = continent_totals.groupby(['Year','Group'], as_index=False)['Population'].sum()
world_totals = shares.groupby('Year')['Population'].transform('sum')
shares['Share (%)'] = shares['Population'] / world_totals * 100

fig = px.bar(shares, x='Year', y='Share (%)', color='Group', barmode='stack',
             template='plotly_white',
             title='Africa’s share is rising while Europe’s is falling, 1970–2022',
             color_discrete_map={'Africa':'#0072B2','Asia':'#E69F00',
                                 'Europe':'#B0B0B0','Rest of the world':'#999999'},
             labels={'Share (%)':'Share of world population (%)','Year':'Year'})
fig.update_layout(yaxis=dict(range=[0,100], ticksuffix='%', showgrid=False),
                  xaxis=dict(showgrid=False, dtick=10), legend_title_text='',
                  bargap=0.15)
fig.show()